In [1]:
# XGBOOST +mBert

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertModel
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
import torch
import re
import logging
from sklearn.preprocessing import LabelEncoder
from imblearn.combine import SMOTETomek
from collections import Counter

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

# Function to preprocess Tamil text
def preprocess_text(text):
    if not isinstance(text, str):
        logger.warning("Non-string input detected; replacing with empty string.")
        return '' 
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    return text

# Function to load and preprocess the dataset
def load_data(file_path):
    try:
        logger.info(f"Loading dataset from {file_path}...")
        data = pd.read_excel(file_path, engine="openpyxl")
        logger.info(f"Dataset loaded successfully with {len(data)} rows.")
        
        data['content'] = data['content'].fillna('')
        data['labels'] = data['labels'].fillna('unknown')
        
        data['text'] = data['content'].apply(preprocess_text)
        logger.info("Text preprocessing completed.")

        return data['text'], data['labels']
    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        raise

# Function to compute mBERT embeddings
def compute_mbert_embeddings(texts, batch_size=32):
    logger.info("Computing mBERT embeddings...")
    tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model = BertModel.from_pretrained('bert-base-multilingual-cased')
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=512)
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].numpy()  # Using CLS token
            embeddings.extend(batch_embeddings)
            if i % (batch_size * 10) == 0:
                logger.info(f"Processed {i}/{len(texts)} texts...")
    
    embeddings = np.array(embeddings)
    logger.info("mBERT embeddings computed successfully.")
    return embeddings

# Function to balance classes using SMOTE + Tomek Links
def balance_classes(X, y):
    logger.info("Applying SMOTE + Tomek Links to balance classes...")
    smote_tomek = SMOTETomek(random_state=42)
    X_resampled, y_resampled = smote_tomek.fit_resample(X, y)
    logger.info(f"Class distribution after balancing: {Counter(y_resampled)}")
    return X_resampled, y_resampled

# Main function to train and evaluate the model
def train_and_evaluate(train_file, test_file):
    try:
        texts_train, labels_train = load_data(train_file)
        texts_test, labels_test = load_data(test_file)
        
        label_encoder = LabelEncoder()
        labels_train_encoded = label_encoder.fit_transform(labels_train)
        labels_test_encoded = label_encoder.transform(labels_test)
        label_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}
        
        logger.info(f"Class distribution before balancing: {Counter(labels_train_encoded)}")
        
        mbert_embeddings_train = compute_mbert_embeddings(texts_train.tolist())
        mbert_embeddings_test = compute_mbert_embeddings(texts_test.tolist())
        
        X_resampled, y_resampled = balance_classes(mbert_embeddings_train, labels_train_encoded)
        
        logger.info("Training XGBoost classifier...")
        classifier = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
        classifier.fit(X_resampled, y_resampled)
        
        train_predictions = classifier.predict(X_resampled)
        test_predictions = classifier.predict(mbert_embeddings_test)
        
        train_accuracy = accuracy_score(y_resampled, train_predictions)
        test_accuracy = accuracy_score(labels_test_encoded, test_predictions)
        
        logger.info(f"Training Accuracy: {train_accuracy:.4f}")
        logger.info(f"Testing Accuracy: {test_accuracy:.4f}")
        
        print(f"Training Accuracy: {train_accuracy:.4f}")
        print(f"Testing Accuracy: {test_accuracy:.4f}")
        
        logger.info("Classification Report:")
        print(classification_report(labels_test_encoded, test_predictions, target_names=[label_mapping[i] for i in range(len(label_mapping))]))
        
        overall_precision = precision_score(labels_test_encoded, test_predictions, average='weighted', zero_division=0)
        overall_recall = recall_score(labels_test_encoded, test_predictions, average='weighted', zero_division=0)
        overall_f1 = f1_score(labels_test_encoded, test_predictions, average='weighted', zero_division=0)

        print(f"Overall Precision: {overall_precision:.4f}")
        print(f"Overall Recall: {overall_recall:.4f}")
        print(f"Overall F1-Score: {overall_f1:.4f}")
        
    except Exception as e:
        logger.error(f"Error during training and evaluation: {e}")
        raise

# File paths for training and testing datasets
train_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx"
test_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx"

# Run the training and evaluation
train_and_evaluate(train_file, test_file)


2025-04-12 22:33:52,606 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx...
2025-04-12 22:33:53,210 - INFO - Dataset loaded successfully with 4352 rows.
2025-04-12 22:33:53,257 - INFO - Text preprocessing completed.
2025-04-12 22:33:53,257 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx...
2025-04-12 22:33:53,325 - INFO - Dataset loaded successfully with 544 rows.
2025-04-12 22:33:53,331 - INFO - Text preprocessing completed.
2025-04-12 22:33:53,334 - INFO - Class distribution before balancing: Counter({3: 1361, 5: 790, 1: 637, 4: 575, 6: 412, 0: 406, 2: 171})
2025-04-12 22:33:53,334 - INFO - Computing mBERT embeddings...
2025-04-12 22:33:59,114 - INFO - Processed 0/4352 texts...
2025-04-12 22:34:49,476 - INFO - Processed 320/4352 texts...
2025-04-12 22:35:24,232 - INFO - Processed 640/4352 texts...
2025-04-12 22:36:17,002 - INFO - Processed 960/4352 texts...
2025-04-12 22:36:56,669 - INFO - 

Training Accuracy: 0.9882
Testing Accuracy: 0.2904
                   precision    recall  f1-score   support

         Negative       0.13      0.13      0.13        46
          Neutral       0.16      0.13      0.14        70
None of the above       0.95      0.84      0.89        25
      Opinionated       0.36      0.45      0.40       171
         Positive       0.20      0.19      0.19        75
        Sarcastic       0.27      0.24      0.25       106
    Substantiated       0.13      0.12      0.12        51

         accuracy                           0.29       544
        macro avg       0.32      0.30      0.31       544
     weighted avg       0.28      0.29      0.28       544

Overall Precision: 0.2822
Overall Recall: 0.2904
Overall F1-Score: 0.2841


In [3]:
# SVM +mBert

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
import torch
import re
import logging
from sklearn.preprocessing import LabelEncoder
from imblearn.combine import SMOTETomek
from collections import Counter

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

# Function to preprocess Tamil text
def preprocess_text(text):
    if not isinstance(text, str):
        logger.warning("Non-string input detected; replacing with empty string.")
        return '' 
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    return text

# Function to load and preprocess the dataset
def load_data(file_path):
    try:
        logger.info(f"Loading dataset from {file_path}...")
        data = pd.read_excel(file_path, engine="openpyxl")
        logger.info(f"Dataset loaded successfully with {len(data)} rows.")
        
        data['content'] = data['content'].fillna('')
        data['labels'] = data['labels'].fillna('unknown')
        
        data['text'] = data['content'].apply(preprocess_text)
        logger.info("Text preprocessing completed.")

        return data['text'], data['labels']
    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        raise

# Function to compute mBERT embeddings
def compute_mbert_embeddings(texts, batch_size=32):
    logger.info("Computing mBERT embeddings...")
    tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model = BertModel.from_pretrained('bert-base-multilingual-cased')
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=512)
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].numpy()  # Using CLS token
            embeddings.extend(batch_embeddings)
            if i % (batch_size * 10) == 0:
                logger.info(f"Processed {i}/{len(texts)} texts...")
    
    embeddings = np.array(embeddings)
    logger.info("mBERT embeddings computed successfully.")
    return embeddings

# Function to balance classes using SMOTE + Tomek Links
def balance_classes(X, y):
    logger.info("Applying SMOTE + Tomek Links to balance classes...")
    smote_tomek = SMOTETomek(random_state=42)
    X_resampled, y_resampled = smote_tomek.fit_resample(X, y)
    logger.info(f"Class distribution after balancing: {Counter(y_resampled)}")
    return X_resampled, y_resampled

# Main function to train and evaluate the model
def train_and_evaluate(train_file, test_file):
    try:
        texts_train, labels_train = load_data(train_file)
        texts_test, labels_test = load_data(test_file)
        
        label_encoder = LabelEncoder()
        labels_train_encoded = label_encoder.fit_transform(labels_train)
        labels_test_encoded = label_encoder.transform(labels_test)
        label_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}
        
        logger.info(f"Class distribution before balancing: {Counter(labels_train_encoded)}")
        
        mbert_embeddings_train = compute_mbert_embeddings(texts_train.tolist())
        mbert_embeddings_test = compute_mbert_embeddings(texts_test.tolist())
        
        X_resampled, y_resampled = balance_classes(mbert_embeddings_train, labels_train_encoded)
        
        logger.info("Training SVM classifier...")
        classifier = LinearSVC(random_state=42)
        classifier.fit(X_resampled, y_resampled)
        
        train_predictions = classifier.predict(X_resampled)
        test_predictions = classifier.predict(mbert_embeddings_test)
        
        train_accuracy = accuracy_score(y_resampled, train_predictions)
        test_accuracy = accuracy_score(labels_test_encoded, test_predictions)
        
        logger.info(f"Training Accuracy: {train_accuracy:.4f}")
        logger.info(f"Testing Accuracy: {test_accuracy:.4f}")
        
        print(f"Training Accuracy: {train_accuracy:.4f}")
        print(f"Testing Accuracy: {test_accuracy:.4f}")
        
        logger.info("Classification Report:")
        print(classification_report(labels_test_encoded, test_predictions, target_names=[label_mapping[i] for i in range(len(label_mapping))]))
        
        overall_precision = precision_score(labels_test_encoded, test_predictions, average='weighted', zero_division=0)
        overall_recall = recall_score(labels_test_encoded, test_predictions, average='weighted', zero_division=0)
        overall_f1 = f1_score(labels_test_encoded, test_predictions, average='weighted', zero_division=0)

        print(f"Overall Precision: {overall_precision:.4f}")
        print(f"Overall Recall: {overall_recall:.4f}")
        print(f"Overall F1-Score: {overall_f1:.4f}")
        
    except Exception as e:
        logger.error(f"Error during training and evaluation: {e}")
        raise

# File paths for training and testing datasets
train_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx"
test_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx"

# Run the training and evaluation
train_and_evaluate(train_file, test_file)


2025-04-12 22:47:51,737 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx...
2025-04-12 22:47:52,067 - INFO - Dataset loaded successfully with 4352 rows.
2025-04-12 22:47:52,099 - INFO - Text preprocessing completed.
2025-04-12 22:47:52,099 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx...
2025-04-12 22:47:52,203 - INFO - Dataset loaded successfully with 544 rows.
2025-04-12 22:47:52,210 - INFO - Text preprocessing completed.
2025-04-12 22:47:52,210 - INFO - Class distribution before balancing: Counter({3: 1361, 5: 790, 1: 637, 4: 575, 6: 412, 0: 406, 2: 171})
2025-04-12 22:47:52,210 - INFO - Computing mBERT embeddings...
2025-04-12 22:47:55,810 - INFO - Processed 0/4352 texts...
2025-04-12 22:48:38,490 - INFO - Processed 320/4352 texts...
2025-04-12 22:49:03,774 - INFO - Processed 640/4352 texts...
2025-04-12 22:49:40,631 - INFO - Processed 960/4352 texts...
2025-04-12 22:50:05,932 - INFO - 

Training Accuracy: 0.7227
Testing Accuracy: 0.2592
                   precision    recall  f1-score   support

         Negative       0.08      0.13      0.10        46
          Neutral       0.21      0.26      0.23        70
None of the above       0.95      0.84      0.89        25
      Opinionated       0.41      0.24      0.30       171
         Positive       0.21      0.25      0.23        75
        Sarcastic       0.30      0.24      0.27       106
    Substantiated       0.12      0.22      0.16        51

         accuracy                           0.26       544
        macro avg       0.33      0.31      0.31       544
     weighted avg       0.31      0.26      0.27       544

Overall Precision: 0.3056
Overall Recall: 0.2592
Overall F1-Score: 0.2724


In [5]:
# SVM + mBert + L2

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from transformers import BertTokenizer, BertModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import torch
import re
import logging
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from sklearn.decomposition import PCA
from imblearn.combine import SMOTETomek  # ✅ SMOTE Tomek

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

# Preprocessing function
def preprocess_text(text):
    if not isinstance(text, str):
        logger.warning("Non-string input detected; replacing with empty string.")
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    return text

# Load and preprocess dataset
def load_data(file_path):
    try:
        logger.info(f"Loading dataset from {file_path}...")
        data = pd.read_excel(file_path, engine="openpyxl")
        logger.info(f"Dataset loaded successfully with {len(data)} rows.")
        
        data['content'] = data['content'].fillna('')
        data['labels'] = data['labels'].fillna('unknown')
        
        data['text'] = data['content'].apply(preprocess_text)
        logger.info("Text preprocessing completed.")
        
        return data['text'], data['labels']
    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        raise

# Compute mBERT embeddings
def compute_mbert_embeddings(texts, batch_size=32):
    logger.info("Computing mBERT embeddings...")
    tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model = BertModel.from_pretrained('bert-base-multilingual-cased')
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=512)
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].numpy()  # CLS token
            embeddings.extend(batch_embeddings)
            if i % (batch_size * 10) == 0:
                logger.info(f"Processed {i}/{len(texts)} texts...")
    
    embeddings = np.array(embeddings)
    logger.info("mBERT embeddings computed successfully.")
    return embeddings

# Apply PCA
def apply_pca(X_train, X_test, n_components=100):
    logger.info(f"Applying PCA to reduce dimensions to {n_components}...")
    pca = PCA(n_components=n_components)
    X_train_reduced = pca.fit_transform(X_train)
    X_test_reduced = pca.transform(X_test)
    logger.info("PCA applied successfully.")
    return X_train_reduced, X_test_reduced

# Train and evaluate
def train_and_evaluate(train_file, test_file, regularization='l2', C_value=0.1, pca_components=100):
    try:
        texts_train, labels_train = load_data(train_file)
        texts_test, labels_test = load_data(test_file)
        
        label_encoder = LabelEncoder()
        labels_train_encoded = label_encoder.fit_transform(labels_train)
        labels_test_encoded = label_encoder.transform(labels_test)
        label_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}
        
        logger.info(f"Original training class distribution: {Counter(labels_train_encoded)}")
        
        mbert_embeddings_train = compute_mbert_embeddings(texts_train.tolist())
        mbert_embeddings_test = compute_mbert_embeddings(texts_test.tolist())
        
        # Apply SMOTE Tomek Links
        logger.info("Applying SMOTE Tomek Links for class balancing...")
        smote_tomek = SMOTETomek(random_state=42)
        X_resampled, y_resampled = smote_tomek.fit_resample(mbert_embeddings_train, labels_train_encoded)
        logger.info(f"Resampled training class distribution: {Counter(y_resampled)}")
        
        # Apply PCA
        X_train_reduced, X_test_reduced = apply_pca(X_resampled, mbert_embeddings_test, n_components=pca_components)
        
        classifier = LinearSVC(penalty=regularization, C=C_value, class_weight='balanced', dual=False, random_state=42)
        
        # Cross-validation
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(classifier, X_train_reduced, y_resampled, cv=skf)
        logger.info(f"Cross-validation accuracy (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"Cross-validation accuracy (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
        # Final training and test
        classifier.fit(X_train_reduced, y_resampled)
        train_preds = classifier.predict(X_train_reduced)
        test_preds = classifier.predict(X_test_reduced)
        
        train_acc = accuracy_score(y_resampled, train_preds)
        test_acc = accuracy_score(labels_test_encoded, test_preds)
        
        logger.info(f"Training Accuracy: {train_acc:.4f}")
        logger.info(f"Testing Accuracy: {test_acc:.4f}")
        print(f"Training Accuracy: {train_acc:.4f}")
        print(f"Testing Accuracy: {test_acc:.4f}")
        
        print("Classification Report:")
        print(classification_report(labels_test_encoded, test_preds, target_names=[label_mapping[i] for i in range(len(label_mapping))]))
        
        overall_precision = precision_score(labels_test_encoded, test_preds, average='weighted', zero_division=0)
        overall_recall = recall_score(labels_test_encoded, test_preds, average='weighted', zero_division=0)
        overall_f1 = f1_score(labels_test_encoded, test_preds, average='weighted', zero_division=0)

        print(f"Overall Precision: {overall_precision:.4f}")
        print(f"Overall Recall: {overall_recall:.4f}")
        print(f"Overall F1-Score: {overall_f1:.4f}")
    
    except Exception as e:
        logger.error(f"Error during training and evaluation: {e}")
        raise

# === File paths ===
train_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx"
test_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx"

# === Run ===
train_and_evaluate(train_file, test_file, regularization='l2', C_value=0.01, pca_components=100)


2025-04-12 22:57:47,725 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx...
2025-04-12 22:57:47,900 - INFO - Dataset loaded successfully with 4352 rows.
2025-04-12 22:57:47,938 - INFO - Text preprocessing completed.
2025-04-12 22:57:47,938 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx...
2025-04-12 22:57:47,980 - INFO - Dataset loaded successfully with 544 rows.
2025-04-12 22:57:47,987 - INFO - Text preprocessing completed.
2025-04-12 22:57:47,989 - INFO - Original training class distribution: Counter({3: 1361, 5: 790, 1: 637, 4: 575, 6: 412, 0: 406, 2: 171})
2025-04-12 22:57:47,989 - INFO - Computing mBERT embeddings...
2025-04-12 22:57:51,688 - INFO - Processed 0/4352 texts...
2025-04-12 22:58:33,557 - INFO - Processed 320/4352 texts...
2025-04-12 22:58:58,230 - INFO - Processed 640/4352 texts...
2025-04-12 22:59:33,923 - INFO - Processed 960/4352 texts...
2025-04-12 22:59:59,541 - INFO -

Cross-validation accuracy (mean ± std): 0.4051 ± 0.0045


2025-04-12 23:06:19,285 - INFO - Training Accuracy: 0.4336
2025-04-12 23:06:19,285 - INFO - Testing Accuracy: 0.2629


Training Accuracy: 0.4336
Testing Accuracy: 0.2629
Classification Report:
                   precision    recall  f1-score   support

         Negative       0.18      0.30      0.23        46
          Neutral       0.22      0.14      0.17        70
None of the above       0.74      0.92      0.82        25
      Opinionated       0.41      0.17      0.24       171
         Positive       0.23      0.37      0.28        75
        Sarcastic       0.26      0.27      0.27       106
    Substantiated       0.12      0.20      0.15        51

         accuracy                           0.26       544
        macro avg       0.31      0.34      0.31       544
     weighted avg       0.30      0.26      0.26       544

Overall Precision: 0.2996
Overall Recall: 0.2629
Overall F1-Score: 0.2594


In [7]:
# SVM + mBert + L1

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from transformers import BertTokenizer, BertModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import torch
import re
import logging
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from sklearn.decomposition import PCA
from imblearn.combine import SMOTETomek

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

# Preprocessing function
def preprocess_text(text):
    if not isinstance(text, str):
        logger.warning("Non-string input detected; replacing with empty string.")
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    return text

# Load and preprocess dataset
def load_data(file_path):
    try:
        logger.info(f"Loading dataset from {file_path}...")
        data = pd.read_excel(file_path, engine="openpyxl")
        logger.info(f"Dataset loaded successfully with {len(data)} rows.")
        
        data['content'] = data['content'].fillna('')
        data['labels'] = data['labels'].fillna('unknown')
        
        data['text'] = data['content'].apply(preprocess_text)
        logger.info("Text preprocessing completed.")
        
        return data['text'], data['labels']
    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        raise

# Compute mBERT embeddings
def compute_mbert_embeddings(texts, batch_size=32):
    logger.info("Computing mBERT embeddings...")
    tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model = BertModel.from_pretrained('bert-base-multilingual-cased')
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=512)
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].numpy()  # CLS token
            embeddings.extend(batch_embeddings)
            if i % (batch_size * 10) == 0:
                logger.info(f"Processed {i}/{len(texts)} texts...")
    
    embeddings = np.array(embeddings)
    logger.info("mBERT embeddings computed successfully.")
    return embeddings

# Apply PCA
def apply_pca(X_train, X_test, n_components=100):
    logger.info(f"Applying PCA to reduce dimensions to {n_components}...")
    pca = PCA(n_components=n_components)
    X_train_reduced = pca.fit_transform(X_train)
    X_test_reduced = pca.transform(X_test)
    logger.info("PCA applied successfully.")
    return X_train_reduced, X_test_reduced

# Train and evaluate with L1 regularization
def train_and_evaluate(train_file, test_file, C_value=0.1, pca_components=100):
    try:
        texts_train, labels_train = load_data(train_file)
        texts_test, labels_test = load_data(test_file)
        
        label_encoder = LabelEncoder()
        labels_train_encoded = label_encoder.fit_transform(labels_train)
        labels_test_encoded = label_encoder.transform(labels_test)
        label_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}
        
        logger.info(f"Original training class distribution: {Counter(labels_train_encoded)}")
        
        mbert_embeddings_train = compute_mbert_embeddings(texts_train.tolist())
        mbert_embeddings_test = compute_mbert_embeddings(texts_test.tolist())
        
        # Apply SMOTE Tomek Links
        logger.info("Applying SMOTE Tomek Links for class balancing...")
        smote_tomek = SMOTETomek(random_state=42)
        X_resampled, y_resampled = smote_tomek.fit_resample(mbert_embeddings_train, labels_train_encoded)
        logger.info(f"Resampled training class distribution: {Counter(y_resampled)}")
        
        # Apply PCA
        X_train_reduced, X_test_reduced = apply_pca(X_resampled, mbert_embeddings_test, n_components=pca_components)
        
        # L1 regularization SVM
        classifier = LinearSVC(
            penalty='l1',
            C=C_value,
            class_weight='balanced',
            dual=False,
            random_state=42,
            max_iter=10000
        )
        
        # Cross-validation
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(classifier, X_train_reduced, y_resampled, cv=skf)
        logger.info(f"Cross-validation accuracy (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"Cross-validation accuracy (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
        # Final training and test
        classifier.fit(X_train_reduced, y_resampled)
        train_preds = classifier.predict(X_train_reduced)
        test_preds = classifier.predict(X_test_reduced)
        
        train_acc = accuracy_score(y_resampled, train_preds)
        test_acc = accuracy_score(labels_test_encoded, test_preds)
        
        logger.info(f"Training Accuracy: {train_acc:.4f}")
        logger.info(f"Testing Accuracy: {test_acc:.4f}")
        print(f"Training Accuracy: {train_acc:.4f}")
        print(f"Testing Accuracy: {test_acc:.4f}")
        
        print("\nClassification Report:")
        print(classification_report(labels_test_encoded, test_preds, target_names=[label_mapping[i] for i in range(len(label_mapping))]))
        
        # Additional L1-specific analysis
        print("\nL1 Regularization Analysis:")
        print(f"Number of non-zero coefficients: {np.sum(classifier.coef_ != 0)}")
        print(f"Sparsity ratio: {1.0 - np.mean(classifier.coef_ != 0):.2%}")
        
        # Overall weighted metrics
        overall_precision = precision_score(labels_test_encoded, test_preds, average='weighted', zero_division=0)
        overall_recall = recall_score(labels_test_encoded, test_preds, average='weighted', zero_division=0)
        overall_f1 = f1_score(labels_test_encoded, test_preds, average='weighted', zero_division=0)

        print(f"\nOverall Precision: {overall_precision:.4f}")
        print(f"Overall Recall: {overall_recall:.4f}")
        print(f"Overall F1-Score: {overall_f1:.4f}")

    except Exception as e:
        logger.error(f"Error during training and evaluation: {e}")
        raise

# === File paths ===
train_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx"
test_file = r"C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx"

# === Run with L1 ===
train_and_evaluate(train_file, test_file, C_value=0.01, pca_components=100)


2025-04-12 23:10:16,257 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_train(3).xlsx...
2025-04-12 23:10:16,420 - INFO - Dataset loaded successfully with 4352 rows.
2025-04-12 23:10:16,452 - INFO - Text preprocessing completed.
2025-04-12 23:10:16,452 - INFO - Loading dataset from C:\Users\haris\OneDrive\Desktop\Sentiment Analysis\PS_test.xlsx...
2025-04-12 23:10:16,495 - INFO - Dataset loaded successfully with 544 rows.
2025-04-12 23:10:16,503 - INFO - Text preprocessing completed.
2025-04-12 23:10:16,503 - INFO - Original training class distribution: Counter({3: 1361, 5: 790, 1: 637, 4: 575, 6: 412, 0: 406, 2: 171})
2025-04-12 23:10:16,503 - INFO - Computing mBERT embeddings...
2025-04-12 23:10:19,826 - INFO - Processed 0/4352 texts...
2025-04-12 23:11:00,135 - INFO - Processed 320/4352 texts...
2025-04-12 23:11:26,067 - INFO - Processed 640/4352 texts...
2025-04-12 23:12:01,062 - INFO - Processed 960/4352 texts...
2025-04-12 23:12:26,040 - INFO -

Cross-validation accuracy (mean ± std): 0.3321 ± 0.0031
Training Accuracy: 0.3563
Testing Accuracy: 0.2500

Classification Report:
                   precision    recall  f1-score   support

         Negative       0.23      0.22      0.22        46
          Neutral       0.18      0.10      0.13        70
None of the above       0.44      0.96      0.60        25
      Opinionated       0.40      0.17      0.24       171
         Positive       0.19      0.28      0.23        75
        Sarcastic       0.27      0.35      0.30       106
    Substantiated       0.09      0.16      0.12        51

         accuracy                           0.25       544
        macro avg       0.26      0.32      0.26       544
     weighted avg       0.27      0.25      0.24       544


L1 Regularization Analysis:
Number of non-zero coefficients: 143
Sparsity ratio: 79.57%

Overall Precision: 0.2750
Overall Recall: 0.2500
Overall F1-Score: 0.2390
